In [1]:
suppressPackageStartupMessages(library(Seurat))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(patchwork))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(future))
suppressPackageStartupMessages(library(ggrepel))
suppressPackageStartupMessages(library(lsa))

In [3]:
E8.5_1 <- Read10X(data.dir = "/tscc/projects/ps-gleesonlab8/Uniformly_processed_data/20250819_Harry_NC_scRNAseq_10x_IGM/WNT85/outs/filtered_feature_bc_matrix")
E8.5_2 <- Read10X(data.dir = "/tscc/projects/ps-gleesonlab8/Uniformly_processed_data/20250819_Harry_NC_scRNAseq_10x_IGM/WNT86/outs/filtered_feature_bc_matrix")
E9.0_1 <- Read10X(data.dir = "/tscc/projects/ps-gleesonlab8/Uniformly_processed_data/20250819_Harry_NC_scRNAseq_10x_IGM/WNT90/outs/filtered_feature_bc_matrix")
E9.0_2 <- Read10X(data.dir = "/tscc/projects/ps-gleesonlab8/Uniformly_processed_data/20250819_Harry_NC_scRNAseq_10x_IGM/WNT91/outs/filtered_feature_bc_matrix")
E9.5_1 <- Read10X(data.dir = "/tscc/projects/ps-gleesonlab8/Uniformly_processed_data/20250819_Harry_NC_scRNAseq_10x_IGM/WNT95/outs/filtered_feature_bc_matrix")

In [4]:
df85 <- CreateSeuratObject(counts = E8.5_1, project = "E8.5_1", min.cells = 3, min.features = 200)
saveRDS(df85, file="/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/SeuratObject_WNT85.rds")
df86 <- CreateSeuratObject(counts = E8.5_2, project = "E8.5_2", min.cells = 3, min.features = 200)
saveRDS(df86, file="/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/SeuratObject_WNT86.rds")
df90 <- CreateSeuratObject(counts = E9.0_1, project = "E9.0_1", min.cells = 3, min.features = 200)
saveRDS(df90, file="/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/SeuratObject_WNT90.rds")
df91 <- CreateSeuratObject(counts = E9.0_2, project = "E9.0_2", min.cells = 3, min.features = 200)
saveRDS(df91, file="/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/SeuratObject_WNT91.rds")
df95 <- CreateSeuratObject(counts = E9.5_1, project = "E9.5_1", min.cells = 3, min.features = 200)
saveRDS(df95, file="/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/SeuratObject_WNT95.rds")

In [5]:
merged <- merge(df85, y=c(df86,df90,df91,df95), add.cell.id=c("E8.5_1","E8.5_2","E9.0_1","E9.0_2","E9.5_1"), project = "Wnt-Tdt")

In [ ]:
saveRDS(merged,file="/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/Wnt1-Tdt_merged.rds")

In [ ]:
df0<-readRDS ("/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/Wnt1-Tdt_merged.rds")

In [ ]:
df0[["percent.mt"]] <- PercentageFeatureSet(df0, pattern = "^mt-")
# # check raw data quality
options(repr.plot.width=12, repr.plot.height=6)
VlnPlot(df0, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3, pt.size = 0.06,log = T)

In [ ]:
df0@meta.data$stage[df0@meta.data$orig.ident %in% c('E8.5_1', 'E8.5_2')] <-'E8.5'
df0@meta.data$stage[df0@meta.data$orig.ident %in% c('E9.0_1', 'E9.0_2')] <-'E9.0'
df0@meta.data$stage[df0@meta.data$orig.ident %in% c('E9.5_1')] <-'E9.5'
table(df0@meta.data$stage)

In [ ]:
# # cut off setup for QC
df0 <- subset(df0, subset = nFeature_RNA > 400 & nCount_RNA >500 & percent.mt<10)
options(repr.plot.width=12, repr.plot.height=6)
VlnPlot(df0, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3, pt.size = 0.06,log = T)

In [ ]:
protein_coding <- read.csv("/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/Mouse_Protein_Coding_List.csv", header = T, row.names = 1)
df0 <- df0[protein_coding$gene_name]

In [ ]:
saveRDS(df0,file="/tscc/projects/ps-gleesonlab8/User/kivong/Wnt1-Tdt/Wnt1-Tdt_merged.rds")

In [ ]:
df0 <- RunPCA(df0, npcs = 50, ndims.print = 1:5, nfeatures.print = 10)
options(repr.plot.width=5, repr.plot.height=5)
ElbowPlot(df0, ndims = 100)

In [ ]:
df0
plan(strategy = "multicore")
df0 <- RunUMAP(df0,
#umap.method = '',
n.neighbors = 30,
min.dist = 0.2,
metric = 'correlation',
n.components = 2,
dims = 1:15,
seed.use = 42,
# n.epochs = 100,
verbose = F)
plan(strategy = "sequential")

In [ ]:
df0 <- FindNeighbors(object = df0, reduction = 'pca', dims = 1:15, verbose = F)
df0 <- FindClusters(object = df0, resolution = 0.4)

In [ ]:
NC<-c('Sox10', 'Pax7', 'Foxd3','Zic1','Zic3','Zic5','Msx1','Gdf7','Olig3',
      'Snai1','Dlx5','Pak3','Pdgfra','Sox9','Cdh11','Six1','Eya2','Isl1','Lmx1a','Phox2b','Ascl1','Krt18','Epcam')
options(repr.plot.width=8, repr.plot.height=6)
markers.to.plot1 <- sort(NC)
DotPlot(df01, features = rev(markers.to.plot1), cols = c("PRGn"), dot.scale = 8)+RotatedAxis()

In [ ]:
dfN <-subset(x=df01, idents=c('0','1','3','4','6','8','10','11','14'))

In [ ]:
dfN <- RunPCA(dfN, npcs = 50, ndims.print = 1:5, nfeatures.print = 10)
options(repr.plot.width=5, repr.plot.height=5)
ElbowPlot(dfN, ndims = 100)

In [ ]:
dfN <- FindNeighbors(dfN, dims = 1:12)
dfN <- FindClusters(dfN, resolution = 0.5)
dfN<- RunUMAP(dfN, dims = 1:12,seed=45)

In [ ]:
saveRDS(dfN,file=".../Wnt1-Tdt_subset.rds")

In [ ]:
NC<-c('Sox10', 'Pax7', 'Foxd3','Zic1','Zic3','Zic5','Msx1','Gdf7','Olig3',
      'Snai1','Dlx5','Pak3','Pdgfra','Sox9','Cdh11','Six1','Eya2','Isl1','Lmx1a','Phox2b','Ascl1','Krt18','Hand2')
options(repr.plot.width=8, repr.plot.height=6)
markers.to.plot1 <- sort(NC)
DotPlot(dfN, features = rev(markers.to.plot1), cols = c("PRGn"), dot.scale = 8)+RotatedAxis()

In [ ]:
plan(strategy = "multicore")
DEX <- FindAllMarkers(object = dfN,
# min.diff.pct = 0.3,
slot = 'data',
#features = hvg,
return.thresh = 0.01,
min.pct = 0.1,
logfc.threshold = 0.25,
only.pos = TRUE,
verbose = FALSE)
plan(strategy = "sequential")
DEX <- DEX %>%
filter(p_val_adj < 0.01) %>%
mutate(diff.pct = pct.1 - pct.2)
DEX <- DEX[!(duplicated(DEX$gene)),]

In [ ]:
library(viridis)
top10 <- DEX %>%
group_by(cluster) %>%
top_n(n = 10, wt = diff.pct)
pdf(file = ".../Wnt1-Tdt_subset_DEG_Heatmap.pdf", width = 5, height = 4, useDingbats = F);
DoHeatmap(dfN,disp.min = -1,disp.max = 1,slot = 'data',features = top10$gene,size = 2)+scale_fill_viridis()+theme(text = element_text(size = 3)) +
NoLegend()
dev.off()
write.csv(DEX, file = ".../Wnt1-Tdt_subset_DEG.csv")
write.csv(top10, file = ".../Wnt1-Tdt_subset.DEG_Top10.csv")

In [ ]:
dfNC <-subset(x=dfn, idents=c('0','1','3','4','5','7','8','9','10','11','12'))

In [ ]:
dfNC <- RunPCA(dfNC, npcs = 50, ndims.print = 1:5, nfeatures.print = 10)
options(repr.plot.width=5, repr.plot.height=5)
ElbowPlot(dfNC, ndims = 100)

In [ ]:
dfNC <- FindNeighbors(dfNC, dims = 1:12)
dfNC <- FindClusters(dfNC, resolution = 0.5)
dfNC<- RunUMAP(dfNC, dims = 1:12,seed=45)

In [ ]:
saveRDS(dfNC,file=".../Wnt1-Tdt_NC_subset_20250903.rds")

In [ ]:
preEMTNC<-c('Zic1', 'Zic3','Zic5', 'Pax3','Pax7','Msx1','Olig3','Foxd3')
options(repr.plot.width=5, repr.plot.height=4)
markers.to.plot1 <- sort(preEMTNC)
DotPlot(dfNC, features = rev(markers.to.plot1), cols = c("PRGn"), dot.scale = 8)+RotatedAxis()

In [ ]:
dfNC@meta.data$cell_type <- "NA"
dfNC@meta.data$cell_type[dfNC@meta.data$seurat_clusters %in% c('1','2','6','10')] <- 'pre-EMT NC'
dfNC@meta.data$cell_type[dfNC@meta.data$seurat_clusters %in% c('7')] <- 'sensory NC'
dfNC@meta.data$cell_type[dfNC@meta.data$seurat_clusters %in% c('0','3','5')] <- 'delaminating NC'
dfNC@meta.data$cell_type[dfNC@meta.data$seurat_clusters %in% c('11')] <- 'sympathetic NC'
dfNC@meta.data$cell_type[dfNC@meta.data$seurat_clusters %in% c('4','8','9')] <- 'mesenchyme NC'

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 5)

umap1 <- DimPlot(
  dfNC,
  reduction = "umap",
  pt.size = 0.7,
  group.by = "seurat_clusters",
  #split.by = "stage",
  label = TRUE
)
umap1
ggsave("umap_clusters_split_by_stage.pdf", plot = umap1, width = 13, height = 5, units = "in")

In [ ]:
options(repr.plot.width=13, repr.plot.height=5)
umap2 <-DimPlot(dfNC, reduction = 'umap',pt.size=0.7, group.by =c('cell_type'), split.by =c('stage'),label = F)+scale_color_manual(values = c(
      "delaminating NC" = "#1f77b4",
      "mesenchyme NC"   = "#ff7f0e",
      "pre-EMT NC"       = "#2ca02c",
      "sensory NC"      = "#d62728",
      "sympathetic NC"  = "#9467bd"))
umap2
ggsave("umap_cell_type_split_by_stage.pdf", plot=umap2, width=13, height=5, units="in")

In [ ]:
genes <- c("Zic1", "Pax3", "Sox10", "Meox1", "Pou4f1", "Six1","Ascl1","Phox2b")
present  <- genes[genes %in% rownames(dfNC)]
missing  <- setdiff(genes, present)
if (length(missing) > 0) {
  message("Missing (not in object): ", paste(missing, collapse = ", "))
}
emb <- Embeddings(dfNC, "umap")
xlim_all <- range(emb[,1], na.rm = TRUE)
ylim_all <- range(emb[,2], na.rm = TRUE)

options(repr.plot.width = 16, repr.plot.height = 8)

fp_list <- FeaturePlot(
  dfNC,
  features    = present,
  reduction   = "umap",
  pt.size     = 0.4,
  order       = TRUE,
  min.cutoff  = "q05",
  max.cutoff  = "q95",
  combine     = FALSE
)


fp_list <- lapply(seq_along(fp_list), function(i) {
  p <- fp_list[[i]]
  p +
    scale_color_gradient(
      low = "grey90",  
      high = "red"      
    ) +
    coord_fixed(xlim = xlim_all, ylim = ylim_all, ratio = 1) +
    theme(
      legend.position = "right",
      plot.title = element_text(face = "bold", size = 11)
    )
})

p_grid <- wrap_plots(fp_list, ncol = 4)
p_grid

ggsave(
  filename = "featureplots_umap_NC_markers_red.pdf",
  plot = p_grid,
  width = 17, height = 8, units = "in"
)

In [ ]:
gA <- "Neurod1"; gB <- "Neurog2"; gC <- "Pou4f1"
cells_focus <- NULL  # or: WhichCells(dfNC, idents = c(1,2,5,6,10))

emb <- Embeddings(dfNC, "umap")
xlim_all <- range(emb[,1], na.rm=TRUE); ylim_all <- range(emb[,2], na.rm=TRUE)
pad <- 0.05 * diff(xlim_all); xlim_all <- c(xlim_all[1]-pad, xlim_all[2]+pad)
umap_ratio <- (diff(ylim_all)/diff(xlim_all)) * 0.8

M <- GetAssayData(dfNC, slot="counts")
stopifnot(all(c(gA,gB,gC) %in% rownames(M)))
cells_use <- if (is.null(cells_focus)) colnames(M) else cells_focus

vA <- as.integer(as.vector(M[gA, cells_use, drop=FALSE] > 0))
vB <- as.integer(as.vector(M[gB, cells_use, drop=FALSE] > 0))
vC <- as.integer(as.vector(M[gC, cells_use, drop=FALSE] > 0))

pair_AB <- vA & vB
pair_AC <- vA & vC
pair_BC <- vB & vC
triple  <- vA & vB & vC

status <- rep("Other", length(cells_use))
status[pair_AB] <- paste0(gA,"+",gB)
status[pair_AC] <- paste0(gA,"+",gC)
status[pair_BC] <- paste0(gB,"+",gC)
status[triple]  <- paste0(gA,"+",gB,"+",gC)
status <- factor(status, levels=c(
  "Other",
  paste0(gA,"+",gB), paste0(gA,"+",gC), paste0(gB,"+",gC),
  paste0(gA,"+",gB,"+",gC)
))

coords <- emb[cells_use, , drop=FALSE]
stage_vec <- dfNC$stage[match(cells_use, rownames(dfNC@meta.data))]

df <- data.frame(
  cell   = cells_use,
  UMAP_1 = coords[,1],
  UMAP_2 = coords[,2],
  status = status,
  stage  = stage_vec,
  stringsAsFactors = FALSE
)

scheme <- "magenta"

pal_pairs <- switch(scheme,
  red     = c("#E41A1C","#377EB8","#4DAF4A"),
  blue    = c("#1F77B4","#2CA02C","#FF7F0E"),
  teal    = c("#008B8B","#2E8B57","#20B2AA"),
  magenta = c("#9C27B0","#7B1FA2","#F06292"),
  orange  = c("#FF7F0E","#E66101","#FDB863"),
  c("#1F77B4","#2CA02C","#FF7F0E")  
)
col_AB <- pal_pairs[1]; col_AC <- pal_pairs[2]; col_BC <- pal_pairs[3]
col_ABC <- "#6a3d9a"  

colmap <- setNames(
  c(
    alpha("grey80", 0.3),  
    "#D55E00",             
    "#0072B2",          
    "#009E73",             
    "#CC79A7"              
  ),
  c("Other",
    paste0(gA,"+",gB),
    paste0(gA,"+",gC),
    paste0(gB,"+",gC),
    paste0(gA,"+",gB,"+",gC))
)

p_triplet_overlay <-
  ggplot() +

  geom_point(data = subset(df, status=="Other"),
             aes(UMAP_1, UMAP_2),
             color = colmap["Other"], size = 1.5, stroke = 0) +
  
  geom_point(data = subset(df, status %in% c(paste0(gA,"+",gB),
                                             paste0(gA,"+",gC),
                                             paste0(gB,"+",gC))),
             aes(UMAP_1, UMAP_2, color = status),
             size = 1.5, stroke = 0) +
  
  geom_point(data = subset(df, status == paste0(gA,"+",gB,"+",gC)),
             aes(UMAP_1, UMAP_2),
             color = colmap[paste0(gA,"+",gB,"+",gC)], size = 1.5, stroke = 0) +
  scale_color_manual(values = colmap, drop = FALSE) +
  coord_cartesian(xlim = xlim_all, ylim = ylim_all, expand = FALSE) +
  coord_fixed(ratio = umap_ratio) +
  facet_wrap(~ stage, ncol = 3, scales = "fixed") +
  labs(
    title = paste0("Double-positives among ", gA,", ", gB,", ", gC, " (triple+ highlighted)"),
    x = "UMAP_1", y = "UMAP_2", color = "Pair"
  ) +
  theme_classic(base_size = 11) +
  theme(legend.position = "right")

p_triplet_overlay

     ggsave(
  filename = paste0("triplet_", gA, "_", gB, "_", gC, "_splitByStage_okabeito.pdf"),
  plot     = p_triplet_overlay,
  width    = 11,
  height   = 6.5,
  useDingbats = FALSE
)